# Team Analysis AI — Demo Interactiva

Este notebook demuestra las principales funcionalidades del sistema:

1. **Carga de datos** — Jugadores activos con predicciones ML
2. **Modelo ML** — Predicción de valor futuro con XGBoost segmentado
3. **Buscador de jugadores** — Búsqueda con filtros
4. **Ranking xGrowth** — Jugadores con mayor potencial de crecimiento
5. **Simulación de fichajes** — Optimización tipo knapsack
6. **Análisis de fichajes** — Jugadores similares y precio justo

---

In [ ]:
import sys, warnings
sys.path.insert(0, '.')
warnings.filterwarnings('ignore')

from IPython.display import display, HTML, Markdown
import pandas as pd
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)

from webapp.i18n import format_currency

## 1. Carga de datos desde caché

Cargamos todos los jugadores activos de la temporada actual con sus predicciones ML precalculadas.

In [ ]:
from simulator.transfer_simulator import TransferSimulator

SEASON = "2025-2026"
CLUB = "Athletic Club"

sim = TransferSimulator(club_name=CLUB, season=SEASON, transfer_budget=60)
sim.preload_data(verbose=True)

print(f"\n{'='*60}")
print(f"Temporada:  {SEASON}")
print(f"Club:       {CLUB}")
print(f"Jugadores totales cargados: {len(sim.all_players):,}")
print(f"Plantilla del club:         {len(sim.club_players)}")
print(f"{'='*60}")

### 1.1 Plantilla actual del club

In [ ]:
squad_rows = []
for p in sorted(sim.club_players, key=lambda x: ["GK","DEF","MID","ATT"].index(x.position) if x.position in ["GK","DEF","MID","ATT"] else 4):
    mv = p.market_value or 0
    pv = p.predicted_value or 0
    xg = (pv / mv - 1) if mv > 0 and pv > 0 else 0
    squad_rows.append({
        "Jugador": p.name,
        "Pos": p.position,
        "Edad": p.age,
        "Valor actual": format_currency(mv),
        "Valor predicho": format_currency(pv) if pv else "—",
        "xGrowth": f"{xg:+.1%}" if pv else "—",
    })

df_squad = pd.DataFrame(squad_rows)
display(Markdown(f"### Plantilla de {CLUB} ({len(df_squad)} jugadores)"))
display(df_squad)

## 2. Modelo ML — Métricas del predictor

Cargamos el modelo XGBoost y analizamos sus características.

In [ ]:
from ml.value_predictor import ValuePredictor, SegmentedValuePredictor, SEGMENT_THRESHOLDS, MODELS_DIR
import json

# Global model metrics
metrics_path = MODELS_DIR / f"value_model_{SEASON}.json"
if metrics_path.exists():
    with open(metrics_path) as f:
        metrics = json.load(f)
    display(Markdown(f"### Métricas del modelo global ({SEASON})"))
    metrics_df = pd.DataFrame([{
        "Métrica": k.replace('_', ' ').title(),
        "Valor": f"€{v:.2f}M" if 'rmse' in k or 'mae' in k else (f"{v:.1f}%" if 'ape' in k else v),
    } for k, v in metrics.items() if k not in ('train_seasons', 'val_seasons')])
    display(metrics_df)
else:
    print(f"No se encontraron métricas en {metrics_path}")

# Segment models availability
display(Markdown("### Modelos segmentados disponibles"))
seg_info = []
for seg_name, lo, hi in SEGMENT_THRESHOLDS:
    seg_path = MODELS_DIR / f"value_model_{SEASON}_{seg_name}.joblib"
    seg_metrics_path = seg_path.with_suffix('.json')
    status = "✅ Disponible" if seg_path.exists() else "❌ No entrenado"
    hi_str = f"€{hi/1e6:.0f}M" if hi < float('inf') else "∞"
    row = {"Segmento": seg_name, "Rango": f"€{lo/1e6:.0f}M – {hi_str}", "Estado": status}
    if seg_metrics_path.exists():
        with open(seg_metrics_path) as f:
            sm = json.load(f)
        row["Val MAE"] = f"€{sm.get('val_mae', '?')}M"
        row["Val MdAPE"] = f"{sm.get('val_mdape', '?')}%"
    seg_info.append(row)
display(pd.DataFrame(seg_info))

### 2.1 Top-15 features más importantes del modelo

In [ ]:
predictor = ValuePredictor(MODELS_DIR / f"value_model_{SEASON}.joblib")
importances = predictor.get_feature_importance()
top_features = sorted(importances.items(), key=lambda x: x[1], reverse=True)[:15]

fi_df = pd.DataFrame(top_features, columns=["Feature", "Importancia"])
fi_df["Importancia"] = fi_df["Importancia"].apply(lambda x: f"{x:.4f}")
fi_df.index = range(1, len(fi_df) + 1)
display(fi_df)

## 3. Buscador de jugadores

Busca jugadores en toda la base de datos con filtros de posición, edad y valor.

In [ ]:
def search_players(query="", position=None, min_value_m=None, max_value_m=None,
                   min_age=None, max_age=None, limit=20):
    """Search players in the loaded data."""
    results = []
    q = query.lower()
    for p in sim.all_players:
        if q and q not in (p.name or "").lower():
            continue
        if position and (p.position or "") != position:
            continue
        mv = (p.market_value or 0) / 1e6
        if min_value_m and mv < min_value_m:
            continue
        if max_value_m and mv > max_value_m:
            continue
        age = p.age or 0
        if min_age and age < min_age:
            continue
        if max_age and age > max_age:
            continue
        results.append(p)
    results.sort(key=lambda x: x.market_value or 0, reverse=True)
    rows = []
    for p in results[:limit]:
        mv = p.market_value or 0
        pv = p.predicted_value
        xg = (pv / mv - 1) if pv and mv > 0 else None
        rows.append({
            "Jugador": p.name, "Equipo": p.team or "?", "Pos": p.position,
            "Edad": p.age, "Nac.": p.nationality or "?",
            "Valor": format_currency(mv),
            "Predicción": format_currency(pv) if pv else "—",
            "xGrowth": f"{xg:+.1%}" if xg is not None else "—",
        })
    return pd.DataFrame(rows)

display(Markdown("### Ejemplo: Mediapuntas jóvenes (<24 años) de alto valor (>€20M)"))
display(search_players(position="MID", min_value_m=20, max_age=24))

In [ ]:
display(Markdown("### Ejemplo: Delanteros con valor entre €5M y €30M"))
display(search_players(position="ATT", min_value_m=5, max_value_m=30, limit=15))

## 4. Ranking xGrowth — Jugadores con mayor crecimiento proyectado

El **xGrowth** mide el ratio `(valor_predicho / valor_mercado) - 1`. Un xGrowth de +50% significa que el modelo predice que el jugador valdrá un 50% más en 1 año.

In [ ]:
club_ids = {p.player_id for p in sim.club_players}
pool = [
    p for p in sim.all_players
    if p.predicted_value is not None
    and (p.market_value or 0) > 500_000
    and not p.on_loan
    and p.player_id not in club_ids
]

def xgrowth(p):
    return (p.predicted_value / p.market_value) - 1 if p.market_value else 0

top_xg = sorted(pool, key=xgrowth, reverse=True)[:25]

xg_rows = []
for i, p in enumerate(top_xg, 1):
    xg = xgrowth(p)
    xg_rows.append({
        "#": i,
        "Jugador": p.name,
        "Equipo": p.team or "?",
        "Pos": p.position,
        "Edad": p.age,
        "Valor actual": format_currency(p.market_value),
        "Valor predicho": format_currency(p.predicted_value),
        "xGrowth": f"{xg:+.1%}",
        "Precio justo": format_currency(p.predicted_value),
    })

display(Markdown("### Top 25 jugadores por xGrowth (excl. plantilla del club)"))
display(pd.DataFrame(xg_rows).set_index("#"))

## 5. Simulación de fichajes

Ejecutamos una simulación completa: el sistema selecciona qué jugadores vender y encuentra los fichajes óptimos mediante optimización tipo knapsack.

**Parámetros configurables:**
- **Objetivo**: SMV (valor total), Net Benefit (beneficio neto), ROI, Value Growth, Growth %
- **Velocidad**: local (rápida), fast, standard (mejor resultado)
- **Filtros avanzados**: liga, clubs excluidos, valor mínimo, horizonte temporal

In [ ]:
display(Markdown(f"### Simulación para {CLUB} — Presupuesto: €60M — Objetivo: Net Benefit"))

result = sim.run(
    verbose=True,
    generate_summary=False,
    sell_by_value_decline=True,
    objective="net_benefit",
    sim_speed="fast",
    horizon=1,
)

print(f"\n{'='*60}")
print(f"RESULTADOS DE LA SIMULACIÓN")
print(f"{'='*60}")
print(f"Presupuesto inicial:     €{result.initial_budget}M")
print(f"Ingresos por ventas:     €{result.sales_revenue}M")
print(f"Presupuesto total:       €{result.total_budget}M")
print(f"Coste de fichajes:       €{result.total_signing_cost}M")
print(f"Valor predicho fichajes: {format_currency(result.total_predicted_value)}")
print(f"Formación:               {result.recommended_formation}")
print(f"{'='*60}")

### 5.1 Jugadores vendidos

In [ ]:
sold_rows = []
for sp in result.players_sold:
    p = sp.player
    sold_rows.append({
        "Jugador": p.name,
        "Pos": p.position,
        "Edad": p.age,
        "Valor": format_currency(p.market_value),
        "Destino": sp.destination_team or "Sin comprador",
        "Vendido": "✅" if sp.was_sold else "❌",
    })
display(Markdown(f"### Ventas ({len(result.players_sold)} jugadores)"))
display(pd.DataFrame(sold_rows))

### 5.2 Fichajes recomendados

In [ ]:
buy_rows = []
total_cost = 0
total_pred = 0
for p in result.recommended_signings:
    mv = p.market_value or 0
    pv = p.predicted_value or 0
    xg = (pv / mv - 1) if mv > 0 else 0
    total_cost += mv
    total_pred += pv
    buy_rows.append({
        "Jugador": p.name,
        "Equipo origen": p.team or "?",
        "Pos": p.position,
        "Edad": p.age,
        "Coste": format_currency(mv),
        "Valor predicho (1 año)": format_currency(pv),
        "xGrowth": f"{xg:+.1%}",
        "Precio justo": format_currency(pv),
    })

display(Markdown(f"### Fichajes recomendados ({len(result.recommended_signings)} jugadores)"))
display(pd.DataFrame(buy_rows))

net = total_pred - total_cost
roi = (net / total_cost * 100) if total_cost > 0 else 0
display(Markdown(
    f"**Inversión total:** {format_currency(total_cost)} → "
    f"**Valor predicho:** {format_currency(total_pred)} → "
    f"**Beneficio neto:** {format_currency(net)} "
    f"(**ROI: {roi:+.1f}%**)"
))

## 6. Análisis de fichajes — Jugadores similares

Para cada fichaje recomendado, encontramos alternativas financieramente similares (mismo puesto, valor de mercado parecido, xGrowth comparable).

In [ ]:
def player_similarity(a, b):
    if (a.position or "") != (b.position or ""):
        return 0.0
    mv_a, mv_b = a.market_value or 1, b.market_value or 1
    age_a, age_b = a.age or 25, b.age or 25
    xg_a = (a.predicted_value or mv_a) / mv_a - 1
    xg_b = (b.predicted_value or mv_b) / mv_b - 1
    val_sim = 1 - min(abs(mv_a - mv_b) / max(mv_a, mv_b), 1)
    age_sim = 1 - min(abs(age_a - age_b) / 10, 1)
    xg_sim = 1 - min(abs(xg_a - xg_b) / max(abs(xg_a) + 0.01, abs(xg_b) + 0.01), 1)
    return 0.35 * val_sim + 0.25 * age_sim + 0.40 * xg_sim

signing_ids = {p.player_id for p in result.recommended_signings}

for s in result.recommended_signings[:5]:
    xg_s = xgrowth(s)
    display(Markdown(
        f"### {s.name} ({s.position}, {format_currency(s.market_value)}) — "
        f"xGrowth: **{xg_s:+.1%}** — Precio justo: **{format_currency(s.predicted_value)}**"
    ))
    candidates = [
        p for p in pool
        if p.player_id != s.player_id
        and p.player_id not in signing_ids
        and (p.position or "") == (s.position or "")
    ]
    scored = sorted(
        [(p, player_similarity(s, p)) for p in candidates],
        key=lambda x: x[1], reverse=True
    )[:5]
    sim_rows = []
    for p, sc in scored:
        sim_rows.append({
            "Alternativa": p.name,
            "Equipo": p.team or "?",
            "Edad": p.age,
            "Valor": format_currency(p.market_value),
            "xGrowth": f"{xgrowth(p):+.1%}",
            "Precio justo": format_currency(p.predicted_value),
            "Similitud": f"{sc:.0%}",
        })
    display(pd.DataFrame(sim_rows))
    print()

## 7. Resumen de IA (opcional)

Si tienes configurada una API key de OpenAI/Anthropic/Gemini, el sistema genera un informe detallado explicando cada decisión de fichaje.

Descomenta y ejecuta la siguiente celda si quieres probar el resumen con IA.

In [ ]:
# Para activar el resumen de IA, descomenta las siguientes líneas
# y proporciona tu API key:

# import os
# os.environ["OPENAI_API_KEY"] = "sk-..."  # Tu API key aquí
#
# from simulator.llm_summarizer import generate_summary_from_result
# summary = generate_summary_from_result(result, provider="openai", language="es")
# if summary:
#     display(Markdown(summary))
# else:
#     print("No se pudo generar el resumen (verifica tu API key)")

print("(Celda desactivada — descomenta para usar resumen de IA)")

## 8. Comparativa multi-objetivo

Ejecutamos la misma simulación con diferentes objetivos para comparar resultados.

In [ ]:
objectives = {
    "smv": "Maximizar valor total (SMV)",
    "net_benefit": "Maximizar beneficio neto",
    "roi": "Maximizar ROI (%)",
    "growth_pct": "Maximizar % crecimiento",
}

comparison = []
for obj_key, obj_label in objectives.items():
    r = sim.run(
        verbose=False,
        generate_summary=False,
        sell_by_value_decline=True,
        objective=obj_key,
        sim_speed="local",
    )
    cost = sum((p.market_value or 0) for p in r.recommended_signings)
    pred = sum((p.predicted_value or 0) for p in r.recommended_signings)
    net = pred - cost
    roi_val = (net / cost * 100) if cost > 0 else 0
    avg_age = sum(p.age or 0 for p in r.recommended_signings) / max(len(r.recommended_signings), 1)
    comparison.append({
        "Objetivo": obj_label,
        "Fichajes": len(r.recommended_signings),
        "Inversión": format_currency(cost),
        "Valor predicho": format_currency(pred),
        "Beneficio neto": format_currency(net),
        "ROI": f"{roi_val:+.1f}%",
        "Edad media": f"{avg_age:.1f}",
    })

display(Markdown(f"### Comparativa de objetivos para {CLUB} (€{sim.budget}M)"))
display(pd.DataFrame(comparison))

---

## Resumen de la demo

| Feature | Descripción |
|---------|-------------|
| **Datos** | +12,000 jugadores activos con valoraciones históricas |
| **ML** | XGBoost con 60+ features, modelos segmentados por rango de valor |
| **xGrowth** | Métrica propia de potencial de crecimiento |
| **Simulador** | Optimización knapsack multi-posición con presupuesto |
| **Objetivos** | SMV, Net Benefit, ROI, Value Growth, Growth % |
| **Análisis** | Jugadores similares, precio justo, resumen IA |
| **Filtros** | Liga, clubs excluidos, valor mínimo, horizonte temporal |
| **Frontend** | Streamlit + React (Vite + Tailwind) |

### Cómo ejecutar la app web

```bash
# Opción 1: Streamlit
streamlit run streamlit_app.py

# Opción 2: React + FastAPI
python scripts/run_demo.py
```